In [207]:

# imports
import json
import pandas as pd
from os import path

# constants
intersection_data_path = ("~/code/county_coverage/data/raw/intersections/" +
                            "Jefferson_County_KY_Street_Intersections.geojson")

In [208]:
def get_records(intersection_data_path):
    with open(intersection_data_path, 'r') as file:
        data = json.load(file)
    features = data['features']
    for feature in features:
        properties = feature['properties']
        geometry = feature['geometry']
        # properties['geo_type'] = geometry['type'] # Always == "Point". Not useful.
        properties['GEOMETRY'] = geometry['coordinates']
        yield properties

df = pd.DataFrame.from_dict(get_records(path.expanduser(intersection_data_path))).convert_dtypes()

bad_row = 100354
# see notes below
# full of nulls that mess up road name compression
# remove row before next step

df = df.drop(df[df.OBJECTID == bad_row].index)

df.head()

,OBJECTID,SIFCODE1,SIFCODE2,INTID,SCCAD_ID,FST_INTPRE,FST_INTNAME,FST_INTSUF,SEC_INTPRE,SEC_INTNAME,SEC_INTSUF,X_COORD,Y_COORD,FST_SIFID,SEC_SIFID,GLOBALID,GEOMETRY
0,1,5464,7662,154647662,1,,REHL,RD,W,REHL,CT,1278243.0,259531.9375,4976,6856,{CD0D9D51-41FB-46FF-B229-AC9C0DDB7E61},"[-85.51044384084946, 38.20588660809318]"
1,2,5464,6551,254646551,2,,REHL,RD,,TUCKER STATION,RD,1273126.375,257588.25,4976,5908,{A9FAED81-F7AE-436F-A657-F95FE1B905E8},"[-85.52816882852979, 38.20037561255241]"
2,3,5464,6551,354646551,3,,REHL,RD,,TUCKER STATION,RD,1273050.50875,257590.85375,4976,5908,{71155AF3-3487-4EDA-8B72-B282378DE7F3},"[-85.52841872335158, 38.200360349057064]"
3,4,3194,9996,431949996,4,,I 64 EAST,,,I 265 RAMP,,1279903.25,265597.5,3076,8763,{AD332CAB-27B0-48B7-ACBF-5CDEEC31654B},"[-85.50495414554669, 38.22260344055154]"
4,5,9349,9996,593499996,5,,I 265 NORTH,,,I 265 RAMP,,1279730.75,265426.4375,8197,8763,{0FE38BAB-8B39-4BE8-BA57-1DAA46FDD4BC},"[-85.50554642815675, 38.222127288727606]"


In [209]:
## find a good index for data

## SCCAD_ID -> No
df.SCCAD_ID.is_unique # False
vc = df.SCCAD_ID.value_counts()
sc2 = vc[vc > 1].index # SCCAD_ID s that point to more than one intersection

df[df.SCCAD_ID.isin(sc2)]


## OBJECTID
df.OBJECTID.is_unique # True


## GLOBALID

  # annoying to look at and use:
  # strings like this: {CD0D9D51-41FB-46FF-B229-AC9C0DDB7E61}

df.GLOBALID.is_unique # True
df.GLOBALID.apply(hash) # negative numbers


## INTID ->  best choice

 # derived from SCCAD_ID + SIFCODES 1 and 2

df.INTID.is_unique # True: can use as index?

df.INTID.str.strip().is_unique # still true

df.INTID.apply(hash) # includes negative numbers
set(''.join(df.INTID.str.strip())) # ->
 # {'0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'C', 'D', 'E', 'F'}
 # only hex chars

 # INTID is a string but it cen be expressed as a number.

df.INTID.apply(lambda x:int(x, base=16)).is_unique # True

"""Conclusion: 
  drop OBJECTID, GLOBALID, SCCAD_ID
  convert INTID to integer and use the result as index"""

df1 = df.drop(['OBJECTID', 'SCCAD_ID', 'GLOBALID'], axis=1)
df1.INTID = df1.INTID.apply(lambda x:int(x, base=16))
df1 = df1.set_index('INTID')
df1.head()

,SIFCODE1,SIFCODE2,FST_INTPRE,FST_INTNAME,FST_INTSUF,SEC_INTPRE,SEC_INTNAME,SEC_INTSUF,X_COORD,Y_COORD,FST_SIFID,SEC_SIFID,GEOMETRY
INTID,,,,,,,,,,,,,
5710837346,5464,7662,,REHL,RD,W,REHL,CT,1278243.0,259531.9375,4976,6856,"[-85.51044384084946, 38.20588660809318]"
10005800273,5464,6551,,REHL,RD,,TUCKER STATION,RD,1273126.375,257588.25,4976,5908,"[-85.52816882852979, 38.20037561255241]"
14300767569,5464,6551,,REHL,RD,,TUCKER STATION,RD,1273050.50875,257590.85375,4976,5908,"[-85.52841872335158, 38.200360349057064]"
18011691414,3194,9996,,I 64 EAST,,,I 265 RAMP,,1279903.25,265597.5,3076,8763,"[-85.50495414554669, 38.22260344055154]"
23945910678,9349,9996,,I 265 NORTH,,,I 265 RAMP,,1279730.75,265426.4375,8197,8763,"[-85.50554642815675, 38.222127288727606]"


In [210]:
# Compress road name info
fst_road_info = df1[["FST_INTPRE", "FST_INTNAME", "FST_INTSUF"]].apply(" ".join, axis=1).str.strip()
sec_road_info = df1[["SEC_INTPRE", "SEC_INTNAME", "SEC_INTSUF"]].apply(" ".join, axis=1).str.strip()

intersections = df1.drop(["FST_INTPRE", "FST_INTNAME", "FST_INTSUF", 
                          "SEC_INTPRE", "SEC_INTNAME", "SEC_INTSUF"], axis=1)
intersections['FST_ROADNAME'] = fst_road_info
intersections['SEC_ROADNAME'] = sec_road_info
intersections.head()


,SIFCODE1,SIFCODE2,X_COORD,Y_COORD,FST_SIFID,SEC_SIFID,GEOMETRY,FST_ROADNAME,SEC_ROADNAME
INTID,,,,,,,,,
5710837346,5464,7662,1278243.0,259531.9375,4976,6856,"[-85.51044384084946, 38.20588660809318]",REHL RD,W REHL CT
10005800273,5464,6551,1273126.375,257588.25,4976,5908,"[-85.52816882852979, 38.20037561255241]",REHL RD,TUCKER STATION RD
14300767569,5464,6551,1273050.50875,257590.85375,4976,5908,"[-85.52841872335158, 38.200360349057064]",REHL RD,TUCKER STATION RD
18011691414,3194,9996,1279903.25,265597.5,3076,8763,"[-85.50495414554669, 38.22260344055154]",I 64 EAST,I 265 RAMP
23945910678,9349,9996,1279730.75,265426.4375,8197,8763,"[-85.50554642815675, 38.222127288727606]",I 265 NORTH,I 265 RAMP


Why are FST_SIFID and SEC_SIFID floats on import?

-> because of a NAN value in item where OBJECTID == 100354
bad_row = 100354

both CSV and JSON are like this

#### CSV:
```csv
X,Y,OBJECTID,SIFCODE1,SIFCODE2,INTID,SCCAD_ID,FST_INTPRE,FST_INTNAME,FST_INTSUF,SEC_INTPRE,SEC_INTNAME,SEC_INTSUF,X_COORD,Y_COORD,FST_SIFID,SEC_SIFID,GLOBALID
```

1227948.0,272835.375,100354,,,,,,,,,,,1227948,272835.375,,,{60D18EC3-BBBC-419B-8A10-62C37F39E987}

#### JSON:
```json
null = float('nan')
{ "type": "Feature",
  "properties": {
     "OBJECTID": 100354, "SIFCODE1": null, "SIFCODE2": null, "INTID": null, "SCCAD_ID": null,
     "FST_INTPRE": null, "FST_INTNAME": null, "FST_INTSUF": null, "SEC_INTPRE": null, "SEC_INTNAME": null,
     "SEC_INTSUF": null, "X_COORD": 1227948.0, "Y_COORD": 272835.375, "FST_SIFID": null, "SEC_SIFID": null,
     "GLOBALID": "{60D18EC3-BBBC-419B-8A10-62C37F39E987}" },
      
  "geometry": { "type": "Point", "coordinates": [ -85.686176099576869, 38.240392433657043 ] } }
```

This interferes with some code that simplifies the road names. Remove the row with nulls and any other before further processing. 


In [211]:
# convert coordinates from LOJIC CRS to (longitude, latitude)
# LOJIC projection: ESRI:102679
# NAD_1983_StatePlane_Kentucky_North_FIPS_1601_Feet

# Standard long, lat: epsg:4326

import numpy as np

from pyproj import CRS
from pyproj.transformer import Transformer

KY_grid_CRS = CRS("ESRI:102679")
longlat_CRS = CRS("epsg:4326")

CRS_transformer = Transformer.from_crs(crs_from=KY_grid_CRS, crs_to=longlat_CRS, always_xy=True).transform

#CRS_transformer.transform(1154395.500000, 188677.437500)


In [225]:
# converting grid point to longitude, latitude

#reindex['coordinates'] = 
XYgrid = intersections.X_COORD.combine(df.Y_COORD, lambda x, y:(x, y))

# have to convert this way because CRS_transformer expects 2 arguments
long_lat_coordinates = intersections.X_COORD.combine(intersections.Y_COORD, CRS_transformer)
long_lat_coordinates = long_lat_coordinates.apply(np.array)
diffs = long_lat_coordinates - intersections.GEOMETRY
diffs


INTID
5710837346           [4.810786933262534e-06, -7.293476116387865e-06]
10005800273        [1.9022999310891464e-05, -2.6813126474678484e-05]
14300767569          [4.815633687371701e-06, -7.291581731294627e-06]
18011691414          [4.810423888557125e-06, -7.296943905998887e-06]
23945910678         [4.810561762269572e-06, -7.2968249327232115e-06]
                                         ...                        
847259462915456       [4.818606214485044e-06, -7.30424124384399e-06]
847265632743817        [4.81256252271578e-06, -7.29362628248964e-06]
847261337735316        [4.81256252271578e-06, -7.29362628248964e-06]
847274257577475      [4.838760816028298e-06, -7.273171526378519e-06]
847272347859222      [4.852362295082457e-06, -7.266027942876008e-06]
Length: 20946, dtype: object

In [258]:
from math import sqrt

west_longitude = -85.94712712079293
east_longitude = -85.3443621648922
south_latitude = 37.99712528351634
north_latitude = 38.38023822809115

delta_long = abs(east_longitude - west_longitude)
delta_lat = abs(north_latitude - south_latitude)

long_dist = 30
lat_dist = 26

def n(point):
    long, lat = point
    v = ((long - west_longitude) / delta_long), ((lat - south_latitude) / delta_lat)
    return np.array(v)

def d(point):
    long, lat = point
    long = (long*long_dist)**2
    lat = (lat*lat_dist)**2
    return sqrt(long + lat)

oft = (long_lat_coordinates.apply(n) - intersections.GEOMETRY.apply(n)).apply(d)*5480

def dd(point):
    long, lat = point
    return (long/delta_long), (lat/delta_lat)

ft = (long_lat_coordinates - intersections.GEOMETRY).apply(dd).apply(d)*5280

ft.describe()
ft[ft>=10]

oft[oft>=100]

INTID
26471278204726      133.990306
92882165092470      104.776528
92912229863543      104.588260
92923046711412      101.480914
362255381312097     174.380952
2696307023537798    142.138332
708179711619108     100.121621
dtype: float64

In [276]:
def haversine(point1, point2):
    """
    Calculate the great circle distance between two points
    on the earth (specified in decimal degrees)
    
    All args must be of equal length.    
    
    """
    lon1, lat1 = map(np.radians, point1)
    lon2, lat2 = map(np.radians, point2)
    
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    
    c = 2 * np.arcsin(np.sqrt(a))
    km = 6378.137 * c
    return km


dd = intersections.GEOMETRY.combine(long_lat_coordinates, haversine)

km_to_miles = 0.6213712

ft = (dd*km_to_miles*5280)
ft[ft>50]

INTID
26471278204726      131.693902
54104598078870       55.677088
92882165092470      103.486683
92912229863543      103.129668
92923046711412       99.911847
92928086495347       92.653856
92937020559475       85.334027
92948816515187       83.848819
92978881286258       86.634784
92991766188145       91.247319
93000356122736       94.584490
93004651090025       84.753044
93466279399528       61.125274
93467774902817       91.712241
93467774931041       91.712241
93468724678753       91.712241
93479164694632       90.466945
93487754563688       86.056873
93496344170594       71.960265
93539294105704       96.183859
93543588810851       82.462392
93552178942056       87.643150
93556473712744       94.857750
93565060763752       94.339999
93568287363125       80.233283
93606952161376       94.447360
281702919518216      59.231054
355640478364215      52.126779
362255381312097     174.228660
407947058451331      66.538407
640141037443477      50.916141
674292192174743      81.748720
68

In [280]:
haversine((east_longitude, north_latitude), (west_longitude, north_latitude))*km_to_miles

haversine((east_longitude, south_latitude), (east_longitude, north_latitude))*km_to_miles

np.float64(26.50020035441295)

In [213]:
geo_diff = (long_lat_coordinates - intersections.GEOMETRY).apply(np.linalg.norm)

intersections.loc[geo_diff[geo_diff > 0.0001].index]

,SIFCODE1,SIFCODE2,X_COORD,Y_COORD,FST_SIFID,SEC_SIFID,GEOMETRY,FST_ROADNAME,SEC_ROADNAME
INTID,,,,,,,,,
2208545531767,3792,6777,1275050.5,270691.5625,12949,6104,"[-85.52219333709645, 38.23642426698946]",LEDGES DR,CREEKVALLEY RD
4840512432473,0506,2959,1274759.0,282274.0625,524,2874,"[-85.52377618828518, 38.26818008336028]",BERRYTOWN RD,HINES RD
26471278204726,5322,5736,1258051.75,289134.875,4861,5212,"[-85.58213654142249, 38.28602260562131]",CRAWLEY CT,MOCKSHIRE DR
54104598078870,3559,6596,1232551.0,266076.1875,3384,13015,"[-85.6695899572017, 38.22201721282951]",KINGS HWY,TYLER LN
92882165092470,D074,D076,1252004.75,321386.125,10124,10126,"[-85.60501649151297, 38.37496056510169]",CHERRY TREE LN,CHERRY TREE CT
...,...,...,...,...,...,...,...,...,...
721352642326681,0000,F099,1205651.78875,241770.70875,1,14426,"[-85.76193115775621, 38.15421172845835]",NO STREET NAME,KENWOOD BUSINESS DR
721507032069444,F259,7544,1247722.86625,232348.13125,14758,6763,"[-85.61535931332455, 38.130043022708435]",AVALON SPRINGS DR,ZELMA FIELDS AVE
721511327036740,F259,7544,1248002.155,232653.7075,14758,6763,"[-85.61453623397385, 38.13091957012409]",AVALON SPRINGS DR,ZELMA FIELDS AVE


In [214]:

#intersections['GEOMETRY'] = long_lat_coordinates
# could encode this as two columns: longitude and latitude
# might still do it, I will typically use this geometry as a point that gets unpacked
# but it seems annoying to have to access two columns constantly when accessing one 
# and unpacking the value is so easy.
# Mirrors the GEOMETRY column in centerlines as well, which is a list of points
intersections.head()

,SIFCODE1,SIFCODE2,X_COORD,Y_COORD,FST_SIFID,SEC_SIFID,GEOMETRY,FST_ROADNAME,SEC_ROADNAME
INTID,,,,,,,,,
5710837346,5464,7662,1278243.0,259531.9375,4976,6856,"[-85.51044384084946, 38.20588660809318]",REHL RD,W REHL CT
10005800273,5464,6551,1273126.375,257588.25,4976,5908,"[-85.52816882852979, 38.20037561255241]",REHL RD,TUCKER STATION RD
14300767569,5464,6551,1273050.50875,257590.85375,4976,5908,"[-85.52841872335158, 38.200360349057064]",REHL RD,TUCKER STATION RD
18011691414,3194,9996,1279903.25,265597.5,3076,8763,"[-85.50495414554669, 38.22260344055154]",I 64 EAST,I 265 RAMP
23945910678,9349,9996,1279730.75,265426.4375,8197,8763,"[-85.50554642815675, 38.222127288727606]",I 265 NORTH,I 265 RAMP


In [215]:
keep_columns = ["FST_ROADNAME", "FST_SIFID",	"SEC_ROADNAME", "SEC_SIFID", "GEOMETRY"]

intersections_clean = intersections[keep_columns].convert_dtypes()
display(intersections_clean.head())

# does not work because INTID has been converted to index and therefore does not exist in columns


,FST_ROADNAME,FST_SIFID,SEC_ROADNAME,SEC_SIFID,GEOMETRY
INTID,,,,,
5710837346,REHL RD,4976,W REHL CT,6856,"[-85.51044384084946, 38.20588660809318]"
10005800273,REHL RD,4976,TUCKER STATION RD,5908,"[-85.52816882852979, 38.20037561255241]"
14300767569,REHL RD,4976,TUCKER STATION RD,5908,"[-85.52841872335158, 38.200360349057064]"
18011691414,I 64 EAST,3076,I 265 RAMP,8763,"[-85.50495414554669, 38.22260344055154]"
23945910678,I 265 NORTH,8197,I 265 RAMP,8763,"[-85.50554642815675, 38.222127288727606]"


In [ ]:

# write transformed data to file.
store_path = "/Users/bencampbell/code/county_coverage/data/cleaner/intersections_data.json"

intersections_clean.to_json(store_path)

def read_in_intersections(path_to_intersection_json):
    out = pd.read_json(path_to_intersection_json)
    # fix some things on import
    out = out.set_index("INTID")
    out.GEOMETRY = out.GEOMETRY.apply(tuple)
    return out             

read_in_intersections(store_path).head()

#store_path_csv = "/Users/bencampbell/code/county_coverage/data/cleaner/intersections_data.csv"
#inter



,FST_ROADNAME,FST_SIFID,SEC_ROADNAME,SEC_SIFID,GEOMETRY
INTID,,,,,
5710837346,REHL RD,4976,W REHL CT,6856,"(-85.5104390301, 38.2058793146)"
10005800273,REHL RD,4976,TUCKER STATION RD,5908,"(-85.52814980550001, 38.2003487994)"
14300767569,REHL RD,4976,TUCKER STATION RD,5908,"(-85.5284139077, 38.2003530575)"
18011691414,I 64 EAST,3076,I 265 RAMP,8763,"(-85.5049493351, 38.2225961436)"
23945910678,I 265 NORTH,8197,I 265 RAMP,8763,"(-85.5055416176, 38.2221199919)"
